# 01 Power Curve Fitting

这个 notebook 展示风机级 SCADA 样例数据如何清洗、预处理并构建经验功率曲线。公开版只使用合成样例数据，字段和处理流程对应真实工程，但不包含真实风机原始数据。

## 1. Data Description

`sample_turbine_scada.csv` 包含 `timestamp`、`turbine_id`、`wind_speed`、`power_kw`、`status`。真实项目中这些字段通常来自 15 分钟 SCADA 报表；公开样例保留字段结构，用于说明功率曲线拟合过程。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from wind_power_baselines.preprocessing import clean_scada
from wind_power_baselines.power_curve import build_power_curve

sns.set_theme(style='whitegrid')
scada = pd.read_csv(PROJECT_ROOT / 'data_sample' / 'sample_turbine_scada.csv')
scada.head()

## 2. Data Cleaning

清洗步骤包括：解析时间、删除重复时间戳、保留 `normal` 状态、剔除负风速/超物理风速、剔除负功率。这样做的原因是停机、限功率和采集异常会把经验功率曲线拉偏。

In [ ]:
cleaned = clean_scada(scada)
print({'raw_rows': len(scada), 'cleaned_rows': len(cleaned), 'removed_rows': len(scada) - len(cleaned)})
cleaned.head()

## 3. EDA Before Fitting

先看风速和功率的散点分布，再看不同风机的样本覆盖。真实项目中如果某台风机某个风速段样本很少，曲线会不稳定，需要在评估里标注。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=cleaned.sample(min(1000, len(cleaned)), random_state=7), x='wind_speed', y='power_kw', hue='turbine_id', s=12, ax=axes[0])
sns.countplot(data=cleaned, x='turbine_id', hue='season', ax=axes[1])
axes[0].set_title('Cleaned wind speed vs power')
axes[1].set_title('Sample coverage by turbine and season')
fig.tight_layout()

## 4. Preprocessing And Curve Construction

公开版采用风速分箱后的中位数功率作为经验功率曲线。中位数比均值更稳健，因为散点里常有停机、限功率、切出和传感器异常。

In [ ]:
curve = build_power_curve(cleaned, 0.5, 12)
curve.to_csv(PROJECT_ROOT / 'data_sample' / 'sample_power_curve.csv', index=False)
curve.head()

## 5. Fitting Result

下图展示每台风机、每个季节的经验功率曲线。第二个点预测 notebook 会直接读取这个曲线表，把预报风速映射为电量。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for (turbine_id, season), group in curve.groupby(['turbine_id', 'season']):
    ax.plot(group['wind_speed_bin'], group['power_kw'], marker='o', label=f'{turbine_id}-{season}')
ax.set_xlabel('Wind speed bin (m/s)')
ax.set_ylabel('Median power (kW)')
ax.set_title('Empirical power curves from cleaned SCADA sample')
ax.legend(fontsize=8)
fig.tight_layout()

## 6. Output

输出文件是 `data_sample/sample_power_curve.csv`，字段为 `turbine_id`、`season`、`wind_speed_bin`、`power_kw`、`sample_count`。它是后续 Baseline 2/3/4 和 AK-D 的风速到功率映射底座。